# Лабораторная работа: Ансамбли моделей машинного обучения

## Цель лабораторной работы
Изучение ансамблей моделей машинного обучения.

## Задание
1. Выбрать набор данных для решения задачи классификации или регресии.
2. Провести удаление или заполнение пропусков и кодирование категориальных признаков.
3. Разделить выборку на обучающую и тестовую.
4. Обучить ансамблевые модели:
- Две модели группы бэггинга
- AdaBoost
- Градиентный бустинг
5. Оценить качество моделей с помощью подходящей метрики и сравнить их.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.filterwarnings('ignore')

# Загрузка набора данных

Для выполнения работы выбран датасет Sick. Этот набор данных используется для задачи бинарной классификации и предсказывает заболевание щитовидной железы. В нем присутствуют как числовые, так и категориальные признаки, а также пропущенные значения.

In [2]:
dataset = fetch_openml(data_id=38, as_frame=True, parser='auto')
X = dataset.data
y = dataset.target

print("Размерность признаков:", X.shape)
print("Размерность целевой переменной:", y.shape)

Размерность признаков: (3772, 29)
Размерность целевой переменной: (3772,)


# Предварительная обработка данных

## Этапы обработки
1. Определение категориальных и числовых признаков
2. Заполнение пропусков средним значением для числовых признаков
3. Заполнение пропусков самым частым значением для категориальных признаков
4. Кодирование категориальных признаков
5. Кодирование целевой переменной

In [ ]:
X_processed = X.copy()

X_processed = X_processed.dropna(axis=1, how='all')

numeric_features = X_processed.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
categorical_features = X_processed.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

if len(numeric_features) > 0:
    num_imputer = SimpleImputer(strategy='mean')
    X_processed[numeric_features] = num_imputer.fit_transform(X_processed[numeric_features])

if len(categorical_features) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X_processed[categorical_features] = cat_imputer.fit_transform(X_processed[categorical_features])
    
    ordinal_encoder = OrdinalEncoder()
    X_processed[categorical_features] = ordinal_encoder.fit_transform(X_processed[categorical_features])

X = X_processed.astype(float)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Обработка завершена.")
print(f"Размерность данных после очистки пустых колонок: {X.shape}")
print(f"Числовых признаков: {len(numeric_features)}")
print(f"Категориальных признаков: {len(categorical_features)}")
print(f"Общее количество пропусков: {X.isna().sum().sum()}")

Обработка завершена.
Размерность данных после очистки пустых колонок: (3772, 28)
Числовых признаков: 6
Категориальных признаков: 22
Общее количество пропусков: 0


# Разделение выборки

Разделяем данные на обучающую и тестовую выборки в пропорции 80 на 20 с помощью метода train_test_split.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

Размер обучающей выборки: (3017, 28)
Размер тестовой выборки: (755, 28)


# Обучение ансамблевых моделей

## Модели группы бэггинга
- BaggingClassifier
- RandomForestClassifier

## Модели бустинга
- AdaBoostClassifier
- GradientBoostingClassifier

In [6]:
# Инициализация моделей с базовыми параметрами
bagging_model = BaggingClassifier(random_state=42)
random_forest_model = RandomForestClassifier(random_state=42)
adaboost_model = AdaBoostClassifier(random_state=42)
gradient_boosting_model = GradientBoostingClassifier(random_state=42)

# Обучение моделей
bagging_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
adaboost_model.fit(X_train, y_train)
gradient_boosting_model.fit(X_train, y_train)

print("Все модели успешно обучены.")

Все модели успешно обучены.


# Оценка качества моделей

Для оценки качества в задаче классификации будем использовать следующие метрики:
- Accuracy
- F1-score

In [8]:
models = {
    "Bagging": bagging_model,
    "Random Forest": random_forest_model,
    "AdaBoost": adaboost_model,
    "Gradient Boosting": gradient_boosting_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-score": f1
    })

results_df = pd.DataFrame(results)
print(results_df)

               Model  Accuracy  F1-score
0            Bagging  0.993377  0.993336
1      Random Forest  0.986755  0.986585
2           AdaBoost  0.980132  0.979745
3  Gradient Boosting  0.986755  0.986916


# Сравнение и анализ результатов

## Результаты оценки
В таблице выше представлены значения метрик точности (Accuracy) и взвешенной F1-меры для каждой из четырех обученных моделей.

## Выводы
1. Сравнение моделей бэггинга: RandomForest и BaggingClassifier обычно показывают схожие результаты, но случайный лес часто работает стабильнее за счет дополнительной рандомизации признаков.
2. Сравнение моделей бустинга: Gradient Boosting чаще всего превосходит AdaBoost на сложных данных, так как оптимизирует произвольные функции потерь.
3. Общий итог: на данном наборе данных (Sick) модели ансамблей показывают высокую точность, так как они эффективно обрабатывают нелинейные зависимости и пропуски в данных после их заполнения.